## Session 8

1. training KNN model

In [ ]:
knn= cv2.ml.KNearest_create()

knn.train(training_data,cv2.ml.ROW_SAMPLE,training_labels)

print("KNN training is complete.")


2.predicit one image

In [ ]:
def predicit_sign(image):
    pixel_row = prepare_image(image)

    #OpenCV wants one sample row
    pixel_row = pixel_row.reshape(1,-1)
    _,result,neighbours,distance = knn.findNearest(pixel_row,k)

    label = int(result[0][0])

    if label == 0:
        prediction="LEFT"
    elif label == 1:
        prediction="RIGHT"
    elif label == 2:
        prediction="STOP"
    else:
        prediction="NO_SIGN"
    return prediction

print("prediction_sign is ready.")
    

3. KNN test without UGOT

In [ ]:
import time
camera = cv2.VideoCapture(0)
if not camera.isOpened():
    raise Exception("Cannot open the laoptop camera.")
print("The camera is on.")

ok,picture_bgr = camera.read()
if not ok:
    raise Exception("No camera data.")

ok,picture_jpg = cv2.imencode(".jpg",picture_bgr)
handle = display(Image(data=picture_jpg.tobytes()),display_id=True)

start_time = time.time()
duration = 40

while time.time()-start_time<duration:
    ok,picture_bgr = camera.read()
    if not ok:
        continue

    BOX_SIZE = 300
    height,width = picture_bgr.shape[:2]
    x1 = width//2 - BOX_SIZE//2
    y1 = height//2 - BOX_SIZE//2
    x2 = x1+BOX_SIZE
    y2 = y1+BOX_SIZE

    crop = picture_bgr[y1:y2,x1:x2]
    prediction = predicit_sign(crop)

    cv2.rectangle(picture_bgr, (x1, y1), (x2, y2), (0, 255, 0), 3)
    cv2.putText(picture_bgr, prediction, (x1, y1 - 15),cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    ok,picture_jpg = cv2.imencode(".jpg",picture_bgr)
    if ok:
        handle.update(Image(data=picture_jpg.tobytes()))

    time.sleep(0.1)

camera.release()
print("Test finished.")